# Mosaic Builder — Visual Decision Demo

This notebook walks through each stage of the pipeline with visual output so you can evaluate whether the decisions are appropriate for the final mosaic.

In [ ]:
import sys
import tempfile
from pathlib import Path

import numpy as np
from IPython.display import HTML, display
from PIL import Image, ImageDraw

# The package is installed into the project's venv via `uv sync`, so a plain
# import should work. The sys.path insert below is only a fallback for
# running this notebook without that install.
try:
    import mosaic_builder  # noqa: F401
except ImportError:
    sys.path.insert(0, str(Path.cwd().parent / "src"))

from mosaic_builder.duckdb_store import (
    ensure_schema,
    get_all_images,
    get_tile_lab_descriptors,
    get_tiles_for_image,
    open_database,
)
from mosaic_builder.ingest import ingest_gallery
from mosaic_builder.matcher import run_match, summarize
from mosaic_builder.renderer import assemble_mosaic, blend_with_target
from mosaic_builder.tiler import (
    delta_e,
    extract_character_tiles,
    iter_gallery_images,
    load_image,
    mean_lab,
    resize_tile,
    texture_score,
)

## Helper: side-by-side display

In [ ]:
import base64
import io


def img_to_data_url(img, max_side=None):
    if max_side and max(img.size) > max_side:
        img = img.copy()
        img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def show_images(images, labels=None, max_side=200):
    """Display a row of images with optional labels."""
    html = '<div style="display:flex; gap:12px; align-items:flex-start; flex-wrap:wrap;">'
    for i, img in enumerate(images):
        url = img_to_data_url(img, max_side=max_side)
        label = labels[i] if labels else ""
        html += f'<div style="text-align:center"><img src="{url}" style="border:1px solid #444; border-radius:4px;"/>'
        if label:
            html += f'<div style="font-size:11px; color:#888; margin-top:4px;">{label}</div>'
        html += "</div>"
    html += "</div>"
    display(HTML(html))


def swatch(r, g, b, size=40):
    """Create a solid color swatch image."""
    return Image.new("RGB", (size, size), (int(r), int(g), int(b)))

## 1. Load gallery images

We use the first 3 images returned by `tiler.iter_gallery_images(GALLERY_DIR)`,
so this cell works against whatever gallery is present without naming any
specific file.

`GALLERY_DIR` points at the `gallery/` symlink, which is not committed to the
repo and will not exist for most people who clone it. **If it is missing, or
holds fewer than 3 images, this cell generates 3 synthetic images with PIL
instead** (and says so explicitly below) so the rest of the notebook still
runs end to end.

In [ ]:
GALLERY_DIR = Path.cwd().parent / "gallery"

gallery_paths = []
if GALLERY_DIR.exists():
    gallery_paths = list(iter_gallery_images(GALLERY_DIR))[:3]

if len(gallery_paths) < 3:
    print(
        f"NOTE: gallery not usable at {GALLERY_DIR} (found {len(gallery_paths)} image(s)) "
        "-- generating 3 synthetic images with PIL instead so this notebook runs anywhere."
    )

    def make_synthetic_image(size, base_color, seed):
        """A solid-ish colour field with random ellipses, so texture_score has something to rank."""
        img = Image.new("RGB", size, base_color)
        draw = ImageDraw.Draw(img)
        rng = np.random.default_rng(seed)
        for _ in range(40):
            x0, y0 = int(rng.integers(0, size[0])), int(rng.integers(0, size[1]))
            w, h = int(rng.integers(10, 60)), int(rng.integers(10, 60))
            jitter = rng.integers(-80, 80, size=3)
            color = tuple(int(v) for v in np.clip(np.array(base_color) + jitter, 0, 255))
            draw.ellipse([x0, y0, x0 + w, y0 + h], fill=color)
        return img

    synth_dir = Path(tempfile.mkdtemp()) / "synthetic_gallery"
    synth_dir.mkdir()
    specs = [
        ("synthetic_wide", (640, 360), (200, 60, 60), 1),
        ("synthetic_tall_a", (360, 640), (60, 160, 60), 2),
        ("synthetic_tall_b", (360, 640), (60, 60, 200), 3),
    ]
    gallery_paths = []
    for name, size, base_color, seed in specs:
        img = make_synthetic_image(size, base_color, seed)
        path = synth_dir / f"{name}.png"
        img.save(path)
        gallery_paths.append(path)
else:
    print(f"Using {len(gallery_paths)} real gallery images from {GALLERY_DIR}")

test_images = {}
for p in gallery_paths:
    img = load_image(p)
    test_images[p.stem] = img

show_images(
    list(test_images.values()),
    [f"{name}\n{img.size[0]}\u00d7{img.size[1]}" for name, img in test_images.items()],
    max_side=250,
)
print(f"Loaded {len(test_images)} gallery images")

## 2. Character tile extraction

For each image, `extract_character_tiles` finds up to 3 maximal square crops, ranked by texture score.

**Key decision:** We take the largest possible square (min of width, height) and slide it to 3 candidate positions (left/center/right or top/center/bottom). The crop with the most texture wins.

This means:
- Wide images get horizontal crops
- Tall images get vertical crops
- Square images get a single crop

In [ ]:
for name, img in test_images.items():
    tiles = extract_character_tiles(img, max_tiles=3)

    # Draw crop boxes on the source
    annotated = img.copy()
    draw = ImageDraw.Draw(annotated)
    colors = ["#ff4444", "#44ff44", "#4444ff"]
    for i, tc in enumerate(tiles):
        draw.rectangle(tc.box, outline=colors[i % 3], width=3)
        draw.text((tc.box[0] + 4, tc.box[1] + 4), f"#{i + 1}", fill=colors[i % 3])

    all_imgs = [annotated] + [tc.image for tc in tiles]
    all_labels = [f"{name} (source)\n{img.size[0]}×{img.size[1]}"] + [
        f"Tile #{i + 1}\nbox={tc.box}\nscore={tc.score:.1f}" for i, tc in enumerate(tiles)
    ]

    print(f"\n{'=' * 60}")
    print(f"{name}: {len(tiles)} tiles extracted")
    show_images(all_imgs, all_labels, max_side=200)

## 3. Resize and observe quality loss

Tiles get resized to a uniform `tile_side` (e.g. 64px). This is what actually gets stored in the DB and used for matching.

**Question to evaluate:** Does the resized tile still look representative of the source region?

In [ ]:
tile_sides = [32, 64, 128]
# Use the first gallery image
demo_name = list(test_images.keys())[0]
demo_img = test_images[demo_name]
tiles = extract_character_tiles(demo_img, max_tiles=1)
raw_tile = tiles[0].image

resized = [resize_tile(raw_tile, s) for s in tile_sides]
show_images(
    [raw_tile] + resized,
    [f"Original crop\n{raw_tile.size[0]}×{raw_tile.size[1]}"]
    + [f"tile_side={s}\n{s}×{s}px" for s in tile_sides],
    max_side=300,
)
print("\nSmaller tile_side = smaller file, faster matching, but less detail.")
print("Larger tile_side = more detail in final mosaic, but more storage + slower matching.")

## 4. RGB vs LAB color descriptors

Each tile stores mean RGB and mean LAB. LAB is perceptually uniform — equal distances in LAB space correspond to equal perceived color differences.

**Why this matters:** When the matcher picks a tile for a grid cell, we want the *perceived* color match to be good, not just the numerical RGB distance.

In [ ]:
# Compare RGB and LAB distances for color pairs that humans perceive differently
color_pairs = [
    ("Reds", (200, 30, 30), (220, 50, 50)),
    ("Red→Green", (200, 30, 30), (30, 200, 30)),
    ("Blues", (30, 30, 200), (50, 50, 220)),
    ("Gray tones", (100, 100, 100), (130, 130, 130)),
    ("Cyan↔Mag", (0, 200, 200), (200, 0, 200)),
]

print(f"{'Pair':<14} {'RGB dist':>10} {'ΔE (LAB)':>10}  Visual")
print("-" * 60)

for pair_name, c1, c2 in color_pairs:
    img1 = Image.new("RGB", (20, 20), c1)
    img2 = Image.new("RGB", (20, 20), c2)

    # Euclidean RGB distance
    rgb_dist = np.sqrt(sum((a - b) ** 2 for a, b in zip(c1, c2, strict=True)))

    # Delta-E LAB distance
    lab1 = mean_lab(img1)
    lab2 = mean_lab(img2)
    de = delta_e(lab1, lab2)

    print(f"{pair_name:<14} {rgb_dist:>10.1f} {de:>10.1f}")

# Show swatches
for pair_name, c1, c2 in color_pairs:
    print(f"\n{pair_name}:")
    show_images([swatch(*c1), swatch(*c2)], [str(c1), str(c2)], max_side=40)

## 5. Texture scoring

Tiles are ranked by texture score (variance + gradient magnitude). Higher scores = more visual interest.

**Decision to evaluate:** Does the scoring correctly prefer textured regions over flat ones?

In [ ]:
# Score all extracted tiles from our gallery images
all_scored = []
for name, img in test_images.items():
    tiles = extract_character_tiles(img, max_tiles=3)
    for i, tc in enumerate(tiles):
        resized = resize_tile(tc.image, 64)
        score = texture_score(resized)
        all_scored.append((f"{name}#{i + 1}", resized, score))

all_scored.sort(key=lambda x: x[2], reverse=True)

print("All tiles ranked by texture score (highest first):")
show_images(
    [img for _, img, _ in all_scored],
    [f"{name}\nscore={score:.1f}" for name, _, score in all_scored],
    max_side=100,
)

## 6. Full ingest → DB round-trip

Run the real pipeline on our 3 gallery images: ingest into DuckDB, read back, and display what was stored.

In [ ]:
import shutil

tmp = tempfile.mkdtemp()
demo_gallery = Path(tmp) / "gallery"
demo_gallery.mkdir()

# Copy just our 3 selected images into a temp gallery for ingestion
for p in gallery_paths:
    shutil.copy2(p, demo_gallery / p.name)

db_path = Path(tmp) / "demo.duckdb"
tile_side = 64

summary = ingest_gallery(demo_gallery, db_path, tile_side=tile_side, show_progress=False)
print(f"Ingested {summary.images} images, {summary.tiles} tiles")
print(f"DB: {db_path}")

In [ ]:
conn = open_database(db_path)
ensure_schema(conn)

images = get_all_images(conn)
for img_row in images:
    tiles = get_tiles_for_image(conn, img_row["image_id"])

    # Load source for comparison
    source = Image.open(img_row["path"])

    # Decode stored tile PNGs
    tile_imgs = [Image.open(io.BytesIO(t["tile_png"])) for t in tiles]
    tile_labels = [
        f"Tile #{t['tile_index'] + 1}\n"
        f"RGB=({t['mean_r']:.0f},{t['mean_g']:.0f},{t['mean_b']:.0f})\n"
        f"LAB=({t['mean_L']:.1f},{t['mean_a']:.1f},{t['mean_bb']:.1f})\n"
        f"score={t['score']:.1f} cov={t['coverage']:.2f}"
        for t in tiles
    ]

    # Show mean-color swatches next to actual tiles
    swatch_imgs = [swatch(t["mean_r"], t["mean_g"], t["mean_b"]) for t in tiles]

    name = Path(img_row["path"]).stem
    print(f"\n{'=' * 60}")
    print(f"{name} ({img_row['width']}×{img_row['height']}) — {len(tiles)} tiles stored")

    # Row 1: source + tiles
    show_images(
        [source] + tile_imgs,
        [f"Source\n{img_row['width']}×{img_row['height']}"] + tile_labels,
        max_side=160,
    )

    # Row 2: mean-color swatches
    print("  Mean color swatches (what the matcher sees):")
    show_images(
        swatch_imgs,
        [f"Tile #{t['tile_index'] + 1} mean RGB" for t in tiles],
        max_side=50,
    )

conn.close()

## 7. Delta-E tile similarity matrix

How similar are the stored tiles to each other? This shows whether the gallery has enough color diversity for a good mosaic.

In [ ]:
conn = open_database(db_path)
descs = get_tile_lab_descriptors(conn)
conn.close()

n = len(descs)
matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        lab_i = (descs[i]["mean_L"], descs[i]["mean_a"], descs[i]["mean_bb"])
        lab_j = (descs[j]["mean_L"], descs[j]["mean_a"], descs[j]["mean_bb"])
        matrix[i, j] = delta_e(lab_i, lab_j)

# Render as a simple text heatmap
tile_names = [f"img{d['image_id']}:t{d['tile_index']}" for d in descs]
header = f"{'':>12}" + "".join(f"{name:>12}" for name in tile_names)
print("Delta-E distance matrix (lower = more similar):")
print(header)
for i, name in enumerate(tile_names):
    row = f"{name:>12}" + "".join(f"{matrix[i, j]:>12.1f}" for j in range(n))
    print(row)

print(f"\nMin non-zero ΔE: {matrix[matrix > 0].min():.1f}")
print(f"Max ΔE: {matrix.max():.1f}")
print(f"Mean ΔE: {matrix[matrix > 0].mean():.1f}")

## 8. Full pipeline: matching + rendering

Now that images are ingested into a database, run the matching and rendering
stages on a real target image: `examples/target_images/target_8bit_checkered_floor_01.png`.

`run_match` divides the target into a grid of `grain`-sized cells, computes
each cell's mean LAB colour, and greedily assigns the nearest tile from the
database. `assemble_mosaic` then pastes those tiles onto a canvas, and
`blend_with_target` can blend the result partway back toward the original
target to improve legibility at a distance.

**The demo database only has the 3 images loaded above (a handful of tiles),
so the mosaic below will look crude and repetitive** -- a real gallery (the
project's own is ~480 photos / ~1400 tiles) gives the matcher far more to
choose from. This section demonstrates the API end to end, not final
quality; see the measured tradeoff table below for numbers from the real
gallery.

In [ ]:
TARGET_PATH = Path.cwd().parent / "examples" / "target_images" / "target_8bit_checkered_floor_01.png"
GRAIN = 64

target_img = load_image(TARGET_PATH)

result = run_match(
    TARGET_PATH,
    db_path,
    grain=GRAIN,
    tile_side=tile_side,
    show_progress=False,
)
print(summarize(result))

conn = open_database(db_path)
ensure_schema(conn)
mosaic_img = assemble_mosaic(result, conn, show_progress=False)
conn.close()

blended_img = blend_with_target(mosaic_img, target_img, alpha=0.3)

show_images(
    [target_img, mosaic_img, blended_img],
    [
        f"Target\n{target_img.size[0]}\u00d7{target_img.size[1]}",
        f"Mosaic (demo DB, grain={GRAIN})\n{mosaic_img.size[0]}\u00d7{mosaic_img.size[1]}",
        "Mosaic blended 30% toward target",
    ],
    max_side=300,
)

## 9. Reuse and repeat-distance tradeoffs

`--max-reuse` caps how many times a single tile can be placed; `--min-repeat-dist`
forbids reusing a tile within a Chebyshev distance of a cell where it was
already placed. Both force more distinct tiles into the mosaic (more visual
variety) at the cost of picking worse colour matches (higher mean delta-E),
because the matcher can no longer always take the single nearest tile.

The cell below runs the same target through `run_match` at three settings and
renders each, so the quality-versus-variety tradeoff is visible directly (on
this tiny 3-image demo database the effect is muted since there are only a
few tiles to begin with -- see the next cell for the measured effect on the
real, ~1400-tile gallery).

In [ ]:
settings = [
    {"label": "unconstrained", "max_reuse": 0, "min_repeat_dist": 0},
    {"label": "max_reuse=2", "max_reuse": 2, "min_repeat_dist": 0},
    {"label": "max_reuse=1, min_repeat_dist=2", "max_reuse": 1, "min_repeat_dist": 2},
]

mosaics = []
labels = []
for s in settings:
    r = run_match(
        TARGET_PATH,
        db_path,
        grain=GRAIN,
        tile_side=tile_side,
        max_reuse=s["max_reuse"],
        min_repeat_dist=s["min_repeat_dist"],
        show_progress=False,
    )
    conn = open_database(db_path)
    ensure_schema(conn)
    m = assemble_mosaic(r, conn, show_progress=False)
    conn.close()
    mosaics.append(m)
    label = s["label"]
    labels.append(f"{label}\n{summarize(r)}")
    print(label, "->", summarize(r))

show_images(mosaics, labels, max_side=250)

### Measured tradeoff on the real gallery

The numbers above come from a 3-image demo database and are not
representative. The table below is measured on the project's real gallery
(480 photos, tile_side 64) matched against the same checkered-floor target:

| Settings | Mean delta-E | Distinct tiles used |
|---|---|---|
| unconstrained | 4.88 | 95 |
| min_repeat_dist 5 | 8.12 | 193 |
| max_reuse 4, min_repeat_dist 3 | 12.00 | 314 |
| max_reuse 2, min_repeat_dist 2 | 13.92 | 536 |
| max_reuse 1 | 17.87 | 1024 |

The project's own rule of thumb puts the "looks off" threshold near a mean
delta-E of 12, so on this gallery pushing past roughly 300 distinct tiles
starts costing visible colour fidelity. This is a direct consequence of the
gallery being tightly clustered in colour space: the mean nearest-neighbour
distance between tiles is only 1.06 (1.46 if tiles from the same photo are
excluded), so there are always many near-duplicate substitutes available
cheaply, but forcing the matcher past that pool of near-duplicates onto
genuinely different tiles rapidly increases the *worst* per-cell error even
while the *average* stays that low headline number above -- 17.87 mean
delta-E at `max_reuse=1` is well past the point where the mismatch reads as
visibly wrong.

## Summary of decisions to evaluate

| Decision | What to look for |
|---|---|
| **Tile extraction** | Do the crop boxes cover the most interesting regions? Are boring areas avoided? |
| **Texture scoring** | Does the ranking match your intuition about visual interest? |
| **Tile resize** | At tile_side=64, is enough detail preserved? Would 32 lose too much? |
| **Mean color swatches** | Does the swatch feel like a fair summary of the tile? |
| **LAB vs RGB distances** | Do the ΔE values better capture perceptual similarity than RGB distance? |
| **Tile diversity** | Is there enough spread in the ΔE matrix for a varied mosaic? |
| **Grid matching (`grain`)** | Does a coarser grain (bigger cells) blur the mosaic too much? Does a finer grain meaningfully improve it, or just slow matching down? |
| **Reuse / repeat-distance constraints** | Does forcing tile variety (lower `--max-reuse`, higher `--min-repeat-dist`) noticeably raise mean ΔE, per the measured tradeoff table above? Is the "~12 ΔE / ~300 distinct tiles" threshold a reasonable place to draw the line for this gallery? |
| **Rendering / blending** | Does the assembled mosaic read correctly at a distance? Does `blend_with_target` improve legibility, and at what alpha does it start to just look like the target again? |